# Model v10: v4-C + surrogato dell'ottimizzatore (SURR) scelto per centrale
Risultati reali di v8/v9: SURR è l'unica idea che funziona (test 0.9438 -> 0.9570, OOF 0.9612 -> 0.9716); kNN, consenso tra casi, calibrazione Platt e FUT non servono e sono stati tolti.
SURR aiuta Tokke/Hogga/Lio ma peggiora Songa/Kjela (e Vinje di poco), quindi per ogni centrale si sceglie C o SURR sull'AUC OOF (mai sul test). Per Vinje il surrogato usa il water value di Totak (non di Vaamarvatn, bacino minuscolo).
Ultima cella: riaddestra su tutti i 2922 casi e scrive la submission Kaggle.

In [2]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import joblib
from functools import cached_property
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view as swv
import yaml
import xgboost as xgb

In [3]:
# create the classes and funcs to prepare the data
def build_cases(X, Y, T=168):
    '''
    Modify the dataset to have them case indexed
    Input:
    - X: x dataset
    - Y: y dataset
    '''
    
    # splitting between daily and hourly features
    hourly = ['day_ahead_price'] + [c for c in X.columns if c.endswith('_inflow')]
    vol = [c for c in X.columns if c.endswith('_volume')]
    wv  = [c for c in X.columns if c.endswith('_wv')]

    # getting the position of every case starting hour
    pos = X.index.get_indexer(Y.index)
    assert (pos >= 0).all() and (pos + T < len(X)).all()

    # creating the sliding windows
    win = swv(X[hourly].to_numpy(np.float32), T, axis=0)       
    xh = win[pos].reshape(len(pos), -1)                        
    
    wv_arr = X[wv].to_numpy()
    xs = np.hstack([
        X[vol].to_numpy()[pos],  # initial volume (s)
        wv_arr[pos],             # initial water value (pos)
        wv_arr[pos + T]          # final water value (pos+T)
    ])

    # getting the names for the columns of x
    hourly_cols = [f"{c}_t{h}" for c in hourly for h in range(T)]
    wv_init_cols = [f"{c}_init" for c in wv]
    wv_final_cols = [f"{c}_final" for c in wv]
    
    all_cols = hourly_cols + vol + wv_init_cols + wv_final_cols

    # excluding run_no e total_calculation_time
    y = pd.DataFrame(
        Y.filter(like='result_committed').to_numpy(np.uint8),
        index=Y.index,
        columns=Y.columns[2:]
    )
    
    x = pd.DataFrame(
        np.hstack([xh, xs]).astype(np.float32),
        index=Y.index,
        columns=all_cols
    )
    
    return x, y


class DataLoader:
    # TODO: Change timestamp data from string to datatime object if available in the data source.
    # TODO: move this in the notebooks (or make it simpler/remove it)

    def __init__(self, data_source=None):
        self.data_source = Path(data_source) if data_source else data_dir

    def _csv(self, folder, filename, date_columns=()):
        file_path = self.data_source / folder / filename
        frame = pd.read_csv(file_path)
        for column in date_columns:
            if column in frame.columns:
                frame[column] = pd.to_datetime(frame[column])
        return frame

    def _yaml(self, folder, filename):
        file_path = self.data_source / folder / filename
        with file_path.open(encoding="utf-8") as stream:
            return yaml.safe_load(stream)

    @cached_property
    def day_ahead_price_data(self):
        return self._csv(
            "kernel",
            "Historical_day_ahead_price_2015_2025.csv",
            ("date",),
        )

    @cached_property
    def inflow_data(self):
        return self._csv("kernel", "Historical_inflow_1958_2025.csv", ("date",))

    @cached_property
    def volume_data(self):
        return self._csv("kernel", "Historical_volume_2015_2024.csv", ("date",))

    @cached_property
    def synthetic_water_value_data(self):
        return self._csv(
            "kernel",
            "Synthetic_water_value_2015_2024.csv",
            ("date",),
        )

    @cached_property
    def commitment_descision_data(self):
        return self._csv("kernel", "Unit_commitment_decisions.csv", ("starttime",))

    @cached_property
    def commitment_decision_data(self):
        return self.commitment_descision_data

    @cached_property
    def min_flow_constraint_data(self):
        return self._csv("extended", "Constraint_min_flow.csv", ("date",))

    @cached_property
    def min_volume_constraint_data(self):
        return self._csv("extended", "Constraint_min_volume.csv", ("date",))

    @cached_property
    def topology_data(self): #TODO: CHECK THIS ONE LATER
        return self._yaml("extended", "Tokke_Vinje_topology.yaml")



# Class to create the entire dataset in one go
class Training_set(DataLoader):
    '''
    It imports all the dataset, merging them using the the index (that will be the date)
    '''
 
    def __init__(self, data_source=None):
        super().__init__(data_source)
        self.y = None
        self.x = None
 
    def load_copies(self):
        sources = {
            "day_ahead_price": self.day_ahead_price_data,
            "inflow": self.inflow_data,
            "volume": self.volume_data,
            "wv": self.synthetic_water_value_data,
            "uc": self.commitment_decision_data,
        }
        return sources
 
 
    def create_x(self):
        '''
        Create the complete X dataset
        
        TODO: feature engineering and selection
                + time split for easier computation (test model on smaller data)
        '''
        
        data = self.load_copies()
        
        # setting the date as index and fixing the columns names
        for name in data.keys():
            if 'date' in list(data[name].columns):
                data[name]['date'] = pd.to_datetime(data[name]['date'], utc=True)
                data[name] = data[name].set_index(data[name]['date']).drop(columns='date')
                if 'Day-ahead price (EUR/MWh)' in data[name].columns:
                    data[name].columns = data[name].columns.str.replace(r'\s*\([^)]*\)', '', regex=True).str.replace(r'[\s-]+', '_', regex=True).str.lower()
                else:
                    data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')
                
            else:
                data[name]['starttime'] = pd.to_datetime(data[name]['starttime'], utc=True)
                data[name] = data[name].set_index(data[name]['starttime']).drop(columns='starttime')
                data[name].columns = data[name].columns.str.lower().str.replace(' ', '_')

        # wv and volume have daily data
        #   need to repet thier value hourly
        hourly = data['day_ahead_price'].index
        for name in ('volume', 'wv'):
            data[name] = data[name].reindex(hourly, method='ffill')
        
        # creating the two datset
        #   y stored for future use
        self.y = data['uc'].copy()
        data.pop('uc')
        x = data['day_ahead_price']
        data.pop('day_ahead_price')
        for name in data.keys():
                x = x.join(data[name].add_suffix('_' + name), how='left')
                
        
        self.x = x.copy()
    
    
    def split_small_data(self, begin_train:str ='2015-01-01', end_train: str='2018-12-31', begin_test: str='2019-01-12', end_test: str='2019-12-31'):
        '''
        Return X and Y dataset
        
        Input:
            - beginnig_train (str, dafault=2015-01-01 ): strart date training, format 'yyyy-mm-dd'
            - beginnig_test (str, dafault=2018-12-31 ): strart date testing, format 'yyyy-mm-dd'
            - end_train (str, dafault=2019-01-01 ): end date of training, format 'yyyy-mm-dd'
            - end_test (str, dafault=2019-12-31 ): end date of testing, format 'yyyy-mm-dd'
        '''
        
        self.create_x()
        X, Y = self.x.copy(), self.y.copy()
        x_train, y_train = build_cases(X, Y.loc[begin_train:end_train])
        x_test,  y_test  = build_cases(X, Y.loc[begin_test:end_test])
        return x_train, y_train, x_test, y_test

In [4]:
DATA = './Users/lucaag/data'
ts = Training_set(data_source=DATA)
ts.create_x()
X_all, Y_all = ts.x.sort_index(), ts.y.sort_index()

split = int(len(Y_all) * 0.8)  # stessa split di v4
x_train, y_train = build_cases(X_all, Y_all.iloc[:split])
x_test, y_test = build_cases(X_all, Y_all.iloc[split + 7:])
topo = ts.topology_data['model']
for x in (x_train, x_test): assert (x.index.to_series().diff().dropna() == pd.Timedelta('1D')).all()  # casi consecutivi
print(x_train.shape, y_train.shape, x_test.shape, y_test.shape)

(2337, 3243) (2337, 2352) (578, 3243) (578, 2352)


## Topologia e target (come v4)

In [5]:
T = 168
RES = {k.lower(): v for k, v in topo['reservoir'].items()}
PL = {k.lower().replace(' ', '_'): v for k, v in topo['plant'].items()}
PN = {k.lower().replace(' ', '_'): g['p_nom'] for k, g in topo['generator'].items()}
TOPO = {
    'vesle_kjela': (['kjelavatn'], 'foersvatn'),
    'kjela': (['foersvatn', 'bordalsvatn'], 'hyljelihyl'),
    'songa': (['songavatn', 'bitdalsvatn'], 'totak'),
    'vinje': (['vaamarvatn', 'totak'], 'vinjevatn'),
    'haukeli': (['vatjern', 'langeidvatn'], 'vinjevatn'),
    'tokke': (['vinjevatn'], 'bandak'),
    'byrte': (['botnedalsvatn'], 'byrtevatn'),
    'lio': (['byrtevatn'], 'bandak'),
    'hogga': (['bandak'], None),
}
UP_PLANTS = {'kjela': ['vesle_kjela'], 'vinje': ['kjela', 'songa'], 'tokke': ['vinje', 'haukeli'],
             'hogga': ['tokke', 'lio'], 'lio': ['byrte']}
SRC = {'vinje': 'totak'}  # bacino di cui SURR usa il water value (default: primo a monte)
ORDER = ['vesle_kjela', 'songa', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']

units = sorted({c[len('result_committed_'):].rsplit('_t', 1)[0] for c in y_train.columns})
plant_units = {p: [i for i, u in enumerate(units) if u.rsplit('_g', 1)[0] == p] for p in ORDER}
ucols = lambda u: [f'result_committed_{units[u]}_t{h}' for h in range(T)]
stack3 = lambda y: np.stack([y[ucols(u)].to_numpy() for u in range(len(units))], axis=1)  # (casi, unità, 168)
Y3, Y3te = stack3(y_train), stack3(y_test)
Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]
out_cols = [c for u in range(len(units)) for c in ucols(u)]
auc = lambda Y, P: roc_auc_score(Y.ravel(), P.ravel())  # micro AUC

## Feature v4

In [6]:
PRICE = [f'day_ahead_price_t{h}' for h in range(T)]

def shift(a, k):
    o = np.roll(a, k, 1)
    if k > 0: o[:, :k] = a[:, :1]
    else: o[:, k:] = a[:, -1:]
    return o

def head(r, v):
    d = RES[r]['vol_head']
    return np.interp(v, d['x'], d['y'])

def static(X, plant):
    up, dn = TOPO[plant]; rs = up + ([dn] if dn else [])
    S = [X[[f'{r}_{s}' for r in rs for s in ('volume', 'wv_init', 'wv_final')]].to_numpy()]
    S += [(X[f'{r}_volume'] / RES[r]['max_vol']).to_numpy()[:, None] for r in rs]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else np.zeros_like(hu)
    wd = X[f'{dn}_wv_final'].to_numpy() if dn else np.zeros(len(X))
    S.append(np.column_stack([hu, hu - np.maximum(hd, PL[plant]['outlet_line']),
                              X[f'{up[0]}_wv_final'].to_numpy() - wd]))
    return np.hstack(S)

def inflow(X, up):  # inflow orario sommato sui bacini a monte (m3/s)
    return sum((X[[f'{r}_inflow_t{h}' for h in range(T)]].to_numpy() for r in up if f'{r}_inflow_t0' in X.columns),
               np.zeros((len(X), T), np.float32))

## Surrogato dell'ottimizzatore
Energia per Mm3 = 2.725·η·H_netta (MWh), quindi la portata massima dell'impianto è `cap = Σp_nom / (2.725·η·H)` in Mm3/h. Per ogni centrale 4 budget d'acqua settimanali: **min** (niente sfioro), **init/final** (ore con prezzo > water value iniziale/finale, limitate tra min e max), **max** (acqua disponibile). Ognuno viene versato nelle ore più care: `unità_t = clip(W/cap − rank_prezzo_t, 0, 1)·n_unità`.

In [7]:
ETA = 0.88

def cap(X, plant):  # portata massima dell'impianto (Mm3/h), stessa testata netta di static()
    up, dn = TOPO[plant]
    hu = head(up[0], X[f'{up[0]}_volume'].to_numpy())
    hd = head(dn, X[f'{dn}_volume'].to_numpy()) if dn else 0
    H = np.maximum(hu - np.maximum(hd, PL[plant]['outlet_line']), 5)
    return sum(PN[units[u]] for u in plant_units[plant]) / (2.725 * ETA * H)

def surr(X, plant, rel=None):  # rel (n,168): rilascio atteso delle centrali a monte (Mm3/h)
    up, c, nu = TOPO[plant][0], cap(X, plant), len(plant_units[plant]); src = SRC.get(plant, up[0])
    V0 = sum(X[f'{r}_volume'].to_numpy() for r in up)
    Win = inflow(X, up).sum(1) * 0.0036 + (rel.sum(1) if rel is not None else 0)  # m3/s * 0.0036 = Mm3/h
    Wmax = np.minimum(V0 + Win, T * c)
    Wmin = np.minimum(np.maximum(V0 + Win - sum(RES[r]['max_vol'] for r in up), 0), Wmax)
    P = X[PRICE].to_numpy(); rank = (-P).argsort(1).argsort(1)
    Wp = [np.clip((P > X[f'{src}_{w}'].to_numpy()[:, None]).sum(1) * c, Wmin, Wmax) for w in ('wv_init', 'wv_final')]
    Ws = [Wmin, *Wp, Wmax]
    S = [np.clip((W / c)[:, None] - rank, 0, 1) * nu for W in Ws]  # unità accese stimate (indice 2 = budget wv_final)
    S += [np.repeat((W / c / T)[:, None], T, 1) for W in Ws]       # frazione di orizzonte a piena capacità
    if rel is not None:
        S += [rel / c[:, None] * nu, np.cumsum(rel, 1) / c[:, None]]  # rilascio a monte in unità equivalenti e in ore cumulate
    return S

def base(X, plant, up_prob=None, rel=None, sur=False):  # matrice (n*168, F); con sur l'ultima colonna è il dispatch stimato
    up, n = TOPO[plant][0], len(X)
    P = X[PRICE].to_numpy(); D = P.reshape(n, 7, 24)
    I = inflow(X, up)
    wvf = X[f'{up[0]}_wv_final'].to_numpy()[:, None]
    h = np.tile(np.arange(T), (n, 1))
    F = [h, h % 24, (X.index.dayofweek.to_numpy()[:, None] + h // 24) % 7,
         P, shift(P, 1), shift(P, -1), P.argsort(1).argsort(1) / (T - 1),
         D.argsort(2).argsort(2).reshape(n, T) / 23, P - np.repeat(D.mean(2), 24, 1),
         P - P.mean(1, keepdims=True), P - wvf, I, np.repeat(I.mean(1, keepdims=True), T, 1)]
    if up_prob is not None:
        F += [up_prob, shift(up_prob, 1), shift(up_prob, 3), np.repeat(up_prob.mean(1, keepdims=True), T, 1)]
    A = np.column_stack([f.ravel() for f in F] + [np.repeat(static(X, plant), T, 0),
                                                  np.repeat(X.index.dayofyear.to_numpy(), T)])
    if sur:
        S = surr(X, plant, rel)
        A = np.column_stack([A] + [s.ravel() for s in S] + [S[2].ravel()])
    return A.astype(np.float32)

## Training: un modello per centrale, OOF a blocchi contigui, stacking topologico (come v4)

In [8]:
NE, FOLDS, GAP, SEEDS = 200, 5, 7, 1  # SEEDS=3 per la submission finale

def mk(seed):
    return xgb.XGBClassifier(n_estimators=NE, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
                             tree_method='hist', n_jobs=-1, random_state=seed)

def fp(X, y, Xs):
    if y.min() == y.max():
        return [np.full(len(x), float(np.clip(y[0], .01, .99))) for x in Xs]
    ms = [mk(s).fit(X, y) for s in range(SEEDS)]
    return [np.mean([m.predict_proba(x)[:, 1] for m in ms], 0) for x in Xs]

def folds(n):
    b = np.linspace(0, n, FOLDS + 1, dtype=int)
    for a, c in zip(b[:-1], b[1:]):
        yield np.r_[0:max(a - GAP, 0), min(c + GAP, n):n], np.arange(a, c)

ridx = lambda c: (np.asarray(c)[:, None] * T + np.arange(T)).ravel()
ylong = lambda c, us: np.concatenate([Yflat[u][ridx(c)] for u in us])

def design(B, c, us, sur):  # righe (caso, ora) x unità; extra: indice unità e, con SURR, (dispatch stimato - indice)
    r = ridx(c); R = B[r]
    def blk(j, u):
        e = [np.full(len(r), u), np.full(len(r), j)] + ([R[:, -1] - j] if sur else [])
        return np.column_stack([R] + e)
    return np.vstack([blk(j, u) for j, u in enumerate(us)])

def fit_group(B, Bt, us, sur):
    n, nt = len(B) // T, len(Bt) // T
    allc = np.arange(n)
    te = fp(design(B, allc, us, sur), ylong(allc, us), [design(Bt, np.arange(nt), us, sur)])[0].reshape(len(us), nt, T)
    oo = np.zeros((len(us), n, T))
    for tr, va in folds(n):
        p = fp(design(B, tr, us, sur), ylong(tr, us), [design(B, va, us, sur)])[0]
        oo[:, va] = p.reshape(len(us), len(va), T)
    return oo, te

def run(SUR, xtr, xte):  # SUR: {centrale: True/False}; ritorna predizioni test e OOF, entrambe (casi, unità, 168)
    oof, te, rtr, rte = {}, {}, {}, {}
    P = np.zeros((len(xte), len(units), T), np.float32); O = np.zeros((len(xtr), len(units), T), np.float32)
    for plant in ORDER:
        us, ups, sur = plant_units[plant], UP_PLANTS.get(plant, []), SUR[plant]
        m = lambda d: np.mean([d[p] for p in ups], 0) if ups else None
        s = lambda d: sum(d[p] for p in ups) if (ups and sur) else None  # rilascio a monte (Mm3/h)
        oo, t = fit_group(base(xtr, plant, m(oof), s(rtr), sur), base(xte, plant, m(te), s(rte), sur), us, sur)
        P[:, us, :], O[:, us, :] = t.transpose(1, 0, 2), oo.transpose(1, 0, 2)
        te[plant], oof[plant] = t.mean(0), oo.mean(0)
        rtr[plant], rte[plant] = oof[plant] * cap(xtr, plant)[:, None], te[plant] * cap(xte, plant)[:, None]
        print(plant, 'ok')
    return P, O

## Confronto C vs SURR e scelta per centrale (sull'OOF)
`SEEDS = 3` solo per la submission finale: qui basta 1.

In [9]:
preds = {n: run({p: v for p in ORDER}, x_train, x_test) for n, v in (('C (v4)', False), ('+SURR', True))}
(Pc, Oc), (Ps, Os) = preds['C (v4)'], preds['+SURR']
rows = [(p, auc(Y3[:, u], Oc[:, u]), auc(Y3[:, u], Os[:, u]), auc(Y3te[:, u], Pc[:, u]), auc(Y3te[:, u], Ps[:, u]))
        for p, u in plant_units.items()]
tab = pd.DataFrame(rows, columns=['centrale', 'oof C', 'oof SURR', 'test C', 'test SURR']).set_index('centrale')
SUR = (tab['oof SURR'] > tab['oof C']).to_dict()  # scelta sull'OOF
P, O = Pc.copy(), Oc.copy()
for p, u in plant_units.items():
    if SUR[p]: P[:, u], O[:, u] = Ps[:, u], Os[:, u]
print('SURR usato per:', [p for p in ORDER if SUR[p]])
print(f'C: oof {auc(Y3, Oc):.4f} test {auc(Y3te, Pc):.4f} | SURR: oof {auc(Y3, Os):.4f} test {auc(Y3te, Ps):.4f} | scelta per centrale: oof {auc(Y3, O):.4f} test {auc(Y3te, P):.4f}')
for w in (.3, .5, .7): print('media C/SURR w =', w, f'oof {auc(Y3, w*Os + (1-w)*Oc):.4f} test {auc(Y3te, w*Ps + (1-w)*Pc):.4f}')
tab.round(4)

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
SURR usato per: ['vesle_kjela', 'byrte', 'haukeli', 'kjela', 'lio', 'vinje', 'tokke', 'hogga']
C: oof 0.9612 test 0.9438 | SURR: oof 0.9716 test 0.9570 | scelta per centrale: oof 0.9716 test 0.9585
media C/SURR w = 0.3 oof 0.9676 test 0.9534
media C/SURR w = 0.5 oof 0.9696 test 0.9555
media C/SURR w = 0.7 oof 0.9709 test 0.9566


,oof C,oof SURR,test C,test SURR
centrale,,,,
vesle_kjela,0.9891,0.9913,0.9854,0.9899
songa,0.9721,0.9711,0.9635,0.9379
byrte,0.9916,0.9955,0.9900,0.9959
haukeli,0.7739,0.8586,0.8876,0.8894
kjela,0.9603,0.9660,0.9811,0.9728
lio,0.9482,0.9557,0.8020,0.8357
vinje,0.9691,0.9693,0.9542,0.9500
tokke,0.9678,0.9803,0.9323,0.9703
hogga,0.9516,0.9658,0.9498,0.9700


## Submission Kaggle: riaddestra su tutti i 2922 casi con la scelta per centrale e predice i 725 casi di test
Mettere `SEEDS = 3` nella cella di training prima di eseguire. `MAP` = percorso di prediction_mapping.csv.

In [10]:
MAP = f'./Users/lucaag/data/prediction_mapping.csv'  # adattare al percorso reale
mp = pd.read_csv(MAP); mp['starttime'] = pd.to_datetime(mp['starttime'], utc=True)
dummy = pd.DataFrame(0, index=pd.DatetimeIndex(mp['starttime']), columns=['Run No', 'total_calculation_time', *y_train.columns])
x_all, y_all = build_cases(X_all, Y_all)
x_kag, _ = build_cases(X_all, dummy)
assert len(x_all) == 2922 and len(x_kag) == len(mp)
Y3 = stack3(y_all); Yflat = [Y3[:, u, :].ravel() for u in range(len(units))]  # run() usa questi target globali
Pk, _ = run(SUR, x_all, x_kag)
sub = pd.DataFrame(Pk.reshape(len(Pk), -1), columns=out_cols)[list(y_train.columns)]
sub.insert(0, 'Run No', mp['Run No'].to_numpy())
sub.to_csv('./Users/lucaag/data/submission_v10.csv', index=False)
print(sub.shape, sub.iloc[:, 1:].to_numpy().mean().round(3))

vesle_kjela ok
songa ok
byrte ok
haukeli ok
kjela ok
lio ok
vinje ok
tokke ok
hogga ok
(725, 2353) 0.471


In [12]:
ss = pd.read_csv(f'./Users/lucaag/data/sample_submission.csv')  # adattare al percorso reale
norm = lambda c: c.lower().replace(' ', '_')
m = {norm(c): c for c in ss.columns}
sub2 = sub.rename(columns=lambda c: m.get(norm(c), c))
assert set(sub2.columns) == set(ss.columns), (set(ss.columns) ^ set(sub2.columns))
sub2 = sub2.set_index('Run No').loc[ss['Run No']].reset_index()[list(ss.columns)]  # stesso ordine di colonne e di righe
sub2.to_csv('./Users/lucaag/data/submission_v10.csv', index=False)
print(sub2.shape)

(725, 2353)
